# Lane Marking Segmentation — BDD100K Training

Trains a DeepLabV3 (MobileNetV3-Large backbone) to segment lane markings by type and color.

**Classes:** background, single white, double white, single yellow, double yellow, road curb, crosswalk

**Before running:**
- Add dataset with BDD100K images (`bdd100k-images-100k`) via the Data panel
- Add dataset with rasterized lane masks (`bdd100k-lane-masks`) via the Data panel
- Enable GPU accelerator (Settings → Accelerator → GPU T4)
- Enable Internet (Settings → Internet → On) for pretrained weights download

**Output:** `best_model.pt` in the Output panel — download after training.

In [ ]:
# ── Cell 1: Environment ───────────────────────────────────────────────────────
import torch
import torchvision
print(f'PyTorch     : {torch.__version__}')
print(f'TorchVision : {torchvision.__version__}')
print(f'CUDA        : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU         : {torch.cuda.get_device_name(0)}')
    print(f'VRAM        : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
# ── Cell 2: Locate datasets ───────────────────────────────────────────────────
from pathlib import Path

WORK = Path('/kaggle/working')
IMG_DIR = Path('/kaggle/input/datasets/burtsbeezer/bdd100k/bdd100k_images_100k/100k')
MASK_DIR = Path('/kaggle/input/datasets/burtsbeezer/bdd100k/bdd100k_lane_masks/lane_masks')

import os
print(f'Images dir : {IMG_DIR}')
print(f'Masks dir  : {MASK_DIR}')
print(f'Train      : {len(os.listdir(IMG_DIR / "train")):,} images, {len(os.listdir(MASK_DIR / "train")):,} masks')
print(f'Val        : {len(os.listdir(IMG_DIR / "val")):,} images, {len(os.listdir(MASK_DIR / "val")):,} masks')

In [ ]:
# ── Cell 3: Dataset + Model ───────────────────────────────────────────────────
import time
import numpy as np
import cv2
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.models.segmentation import deeplabv3_mobilenet_v3_large

NUM_CLASSES = 7
IMG_SIZE = (512, 256)  # W, H
BATCH_SIZE = 16  # T4 can handle more than local
NUM_EPOCHS = 25
LR = 1e-3

CLASS_NAMES = ["bg", "s_white", "d_white", "s_yellow", "d_yellow", "curb", "xwalk"]

# Upweight yellow classes (key geo signal), downweight background
CLASS_WEIGHTS = torch.tensor([
    0.1,   # 0: background
    1.0,   # 1: single white
    2.0,   # 2: double white
    4.0,   # 3: single yellow
    4.0,   # 4: double yellow
    0.5,   # 5: road curb
    0.5,   # 6: crosswalk
], dtype=torch.float32)


class BDDLaneDataset(Dataset):
    def __init__(self, split, img_dir, mask_dir, img_size=IMG_SIZE):
        self.img_dir = img_dir / split
        self.mask_dir = mask_dir / split
        self.img_size = img_size

        # Use os.listdir (faster than glob on large dirs)
        mask_names = [f for f in os.listdir(self.mask_dir) if f.endswith('.png')]
        mask_names.sort()
        self.samples = []
        for mname in mask_names:
            stem = mname[:-4]  # strip .png
            img_file = self.img_dir / f'{stem}.jpg'
            if img_file.exists():
                self.samples.append((img_file, self.mask_dir / mname))

        print(f'  {split}: {len(self.samples)} samples')

        self.img_transform = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406],
                                 std=[0.229, 0.224, 0.225]),
        ])

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, mask_path = self.samples[idx]
        img = cv2.imread(str(img_path))
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = cv2.resize(img, self.img_size, interpolation=cv2.INTER_LINEAR)

        mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
        mask = cv2.resize(mask, self.img_size, interpolation=cv2.INTER_NEAREST)

        img = self.img_transform(img)
        mask = torch.from_numpy(mask).long()
        return img, mask


def build_model(num_classes=NUM_CLASSES):
    model = deeplabv3_mobilenet_v3_large(weights='DEFAULT')
    model.classifier[-1] = nn.Conv2d(256, num_classes, kernel_size=1)
    if model.aux_classifier is not None:
        model.aux_classifier[-1] = nn.Conv2d(10, num_classes, kernel_size=1)
    return model


print('Dataset and model definitions ready.')

In [ ]:
# ── Cell 4: Load data + build model ───────────────────────────────────────────
print('Loading datasets...')
train_ds = BDDLaneDataset('train', IMG_DIR, MASK_DIR)
val_ds = BDDLaneDataset('val', IMG_DIR, MASK_DIR)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=4, pin_memory=True, drop_last=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=4, pin_memory=True)

# Quick sanity check
img, mask = train_ds[0]
print(f'Image shape: {img.shape}, Mask shape: {mask.shape}')
print(f'Mask classes present: {torch.unique(mask).tolist()}')

print('\nBuilding model...')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = build_model().to(device)

# Count params
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Parameters: {total_params/1e6:.1f}M total, {trainable_params/1e6:.1f}M trainable')
print(f'Device: {device}')

In [ ]:
# ── Cell 5: Training loop ─────────────────────────────────────────────────────
def compute_iou(pred, target, num_classes):
    ious = []
    for c in range(num_classes):
        pred_c = (pred == c)
        target_c = (target == c)
        intersection = (pred_c & target_c).sum().item()
        union = (pred_c | target_c).sum().item()
        ious.append(intersection / union if union > 0 else float('nan'))
    return ious


def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss = 0
    for imgs, masks in loader:
        imgs, masks = imgs.to(device), masks.to(device)
        output = model(imgs)['out']
        if output.shape[2:] != masks.shape[1:]:
            output = nn.functional.interpolate(output, size=masks.shape[1:],
                                               mode='bilinear', align_corners=False)
        loss = criterion(output, masks)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)


@torch.no_grad()
def validate(model, loader, criterion, device):
    model.eval()
    total_loss = 0
    all_ious = [[] for _ in range(NUM_CLASSES)]

    for imgs, masks in loader:
        imgs, masks = imgs.to(device), masks.to(device)
        output = model(imgs)['out']
        if output.shape[2:] != masks.shape[1:]:
            output = nn.functional.interpolate(output, size=masks.shape[1:],
                                               mode='bilinear', align_corners=False)
        total_loss += criterion(output, masks).item()

        preds = output.argmax(dim=1)
        for b in range(preds.shape[0]):
            ious = compute_iou(preds[b].cpu(), masks[b].cpu(), NUM_CLASSES)
            for c, iou in enumerate(ious):
                if not np.isnan(iou):
                    all_ious[c].append(iou)

    avg_loss = total_loss / len(loader)
    class_ious = {CLASS_NAMES[c]: np.mean(all_ious[c]) if all_ious[c] else float('nan')
                  for c in range(NUM_CLASSES)}
    mean_iou = np.nanmean([v for v in class_ious.values() if not np.isnan(v)])
    return avg_loss, mean_iou, class_ious


# Setup
weights = CLASS_WEIGHTS.to(device)
criterion = nn.CrossEntropyLoss(weight=weights)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)

best_iou = 0
history = []

print(f'Training for {NUM_EPOCHS} epochs')
print(f'Train: {len(train_ds)}, Val: {len(val_ds)}, Batch: {BATCH_SIZE}')
print('=' * 90)

for epoch in range(NUM_EPOCHS):
    t0 = time.time()

    train_loss = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, mean_iou, class_ious = validate(model, val_loader, criterion, device)
    scheduler.step()

    elapsed = time.time() - t0
    lr = optimizer.param_groups[0]['lr']

    iou_str = ' | '.join(f'{k}:{v:.3f}' if not np.isnan(v) else f'{k}:---'
                         for k, v in class_ious.items() if k != 'bg')

    print(f'Epoch {epoch+1:2d}/{NUM_EPOCHS} | '
          f'train: {train_loss:.4f} | val: {val_loss:.4f} | '
          f'mIoU: {mean_iou:.4f} | lr: {lr:.6f} | {elapsed:.0f}s')
    print(f'  {iou_str}')

    history.append({
        'epoch': epoch + 1, 'train_loss': train_loss, 'val_loss': val_loss,
        'mean_iou': mean_iou, **{f'iou_{k}': v for k, v in class_ious.items()},
    })

    if mean_iou > best_iou:
        best_iou = mean_iou
        torch.save({
            'model': model.state_dict(),
            'epoch': epoch,
            'best_iou': best_iou,
            'class_ious': class_ious,
            'img_size': IMG_SIZE,
            'num_classes': NUM_CLASSES,
        }, WORK / 'best_model.pt')
        print(f'  ★ New best (mIoU: {best_iou:.4f})')

    if (epoch + 1) % 5 == 0:
        torch.save({'model': model.state_dict(), 'epoch': epoch},
                   WORK / f'checkpoint_epoch_{epoch+1}.pt')

print(f'\nDone. Best mIoU: {best_iou:.4f}')

In [ ]:
# ── Cell 6: Training curves + results ─────────────────────────────────────────
import matplotlib.pyplot as plt
import shutil
import pandas as pd

# Save history CSV
df = pd.DataFrame(history)
df.to_csv(WORK / 'training_history.csv', index=False)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Loss
axes[0].plot(df['epoch'], df['train_loss'], label='train')
axes[0].plot(df['epoch'], df['val_loss'], label='val')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Loss')
axes[0].legend()
axes[0].grid(True)

# mIoU
axes[1].plot(df['epoch'], df['mean_iou'], 'g-o', markersize=3)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('mIoU')
axes[1].set_title(f'Mean IoU (best: {best_iou:.4f})')
axes[1].grid(True)

# Per-class IoU (final epoch)
iou_cols = [c for c in df.columns if c.startswith('iou_') and c != 'iou_bg']
final_ious = df.iloc[-1][iou_cols]
names = [c.replace('iou_', '') for c in iou_cols]
colors = ['white', 'lightgray', 'gold', 'orange', 'steelblue', 'lightgreen']
bars = axes[2].bar(names, final_ious, color=colors, edgecolor='black')
axes[2].set_ylabel('IoU')
axes[2].set_title('Per-Class IoU (final epoch)')
axes[2].grid(True, axis='y')

plt.tight_layout()
plt.savefig(WORK / 'training_results.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'\nSaved training_results.png and training_history.csv')

In [ ]:
# ── Cell 7: Visualize predictions on val samples ──────────────────────────────
from IPython.display import display, Image as IPImage

CLASS_COLORS = np.array([
    [0, 0, 0],        # background
    [255, 255, 255],   # single white
    [200, 200, 200],   # double white
    [255, 255, 0],     # single yellow
    [255, 200, 0],     # double yellow
    [0, 0, 255],       # road curb
    [0, 255, 0],       # crosswalk
], dtype=np.uint8)

model.eval()
fig, axes = plt.subplots(4, 3, figsize=(18, 20))

for row in range(4):
    idx = np.random.randint(len(val_ds))
    img, mask_gt = val_ds[idx]

    with torch.no_grad():
        pred = model(img.unsqueeze(0).to(device))['out']
        if pred.shape[2:] != mask_gt.shape:
            pred = nn.functional.interpolate(pred, size=mask_gt.shape,
                                             mode='bilinear', align_corners=False)
        pred = pred.argmax(dim=1).squeeze().cpu().numpy()

    # Denormalize image for display
    img_np = img.permute(1, 2, 0).numpy()
    img_np = img_np * np.array([0.229, 0.224, 0.225]) + np.array([0.485, 0.456, 0.406])
    img_np = np.clip(img_np * 255, 0, 255).astype(np.uint8)

    gt_vis = CLASS_COLORS[mask_gt.numpy()]
    pred_vis = CLASS_COLORS[pred]

    axes[row, 0].imshow(img_np)
    axes[row, 0].set_title('Input')
    axes[row, 0].axis('off')

    axes[row, 1].imshow(gt_vis)
    axes[row, 1].set_title('Ground Truth')
    axes[row, 1].axis('off')

    axes[row, 2].imshow(pred_vis)
    axes[row, 2].set_title('Prediction')
    axes[row, 2].axis('off')

plt.tight_layout()
plt.savefig(WORK / 'val_predictions.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved val_predictions.png')

In [ ]:
# ── Cell 8: Save outputs ──────────────────────────────────────────────────────
import os

outputs = ['best_model.pt', 'training_results.png', 'training_history.csv', 'val_predictions.png']
for f in outputs:
    p = WORK / f
    if p.exists():
        size = p.stat().st_size
        if size > 1e6:
            print(f'  {f} — {size/1e6:.1f} MB')
        else:
            print(f'  {f} — {size/1e3:.1f} KB')
    else:
        print(f'  {f} — NOT FOUND')

# Also save any checkpoints
for f in sorted(WORK.glob('checkpoint_*.pt')):
    print(f'  {f.name} — {f.stat().st_size/1e6:.1f} MB')

print('\nDownload best_model.pt from the Output panel →')